<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Hought_Transform_Line_Detection_Bridge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
Description: Parametric animation of Hough Transform for Line Detection.
"""

# Install and configure ImageMagick for MoviePy
!apt-get install -y imagemagick
!sed -i 's/none/read,write/g' /etc/ImageMagick-6/policy.xml

import cv2
import numpy as np
import requests
from PIL import Image
from io import BytesIO
from moviepy.config import change_settings
from moviepy.editor import ImageClip, CompositeVideoClip, TextClip
from google.colab import files

# Configure MoviePy to use the correct ImageMagick binary
change_settings({"IMAGEMAGICK_BINARY": "/usr/bin/convert"})

# --- PARAMETERS ---
IMAGE_URL = "https://images.unsplash.com/photo-1449034446853-66c86144b0ad?q=80&w=1280&auto=format&fit=crop"
DURATION = 35
TRANSITION_DURATION = 2.0
CANNY_LOW = 50
CANNY_HIGH = 150
HOUGH_RHO = 1
HOUGH_THETA = np.pi/180
HOUGH_THRESHOLD = 100
WATERMARK_TEXT = "@craftsandengineering"
AUTHOR_TEXT = "Created by Mugambi Ndwiga"
OUTPUT_FILE = "hough_transform_demo.mp4"

# --- ASSET PREPARATION ---
def prepare_assets():
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(IMAGE_URL, headers=headers)
    if response.status_code != 200:
        raise Exception(f"Failed to download image. Status: {response.status_code}")

    img = Image.open(BytesIO(response.content)).convert('RGB')
    img = np.array(img)
    img = cv2.resize(img, (1280, 720))

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, CANNY_LOW, CANNY_HIGH)
    edges_colored = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)

    lines = cv2.HoughLinesP(edges, HOUGH_RHO, HOUGH_THETA, HOUGH_THRESHOLD,
                            minLineLength=50, maxLineGap=10)

    line_img = np.zeros_like(img)
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            cv2.line(line_img, (x1, y1), (x2, y2), (0, 255, 0), 2)

    overlay_img = cv2.addWeighted(img, 0.7, line_img, 1.0, 0)
    return img, edges_colored, overlay_img

try:
    orig, edge_view, result = prepare_assets()

    # --- VIDEO CONSTRUCTION ---
    title = TextClip("Hough Transform:\nDetecting the World's Geometry", fontsize=70, color='white', size=(1280, 720), bg_color='black', method='caption')\
        .set_duration(4).set_fps(24).crossfadeout(TRANSITION_DURATION)

    clip1 = ImageClip(orig).set_duration(8).crossfadein(TRANSITION_DURATION).crossfadeout(TRANSITION_DURATION)
    clip2 = ImageClip(edge_view).set_duration(8).crossfadein(TRANSITION_DURATION).crossfadeout(TRANSITION_DURATION)
    clip3 = ImageClip(result).set_duration(10).crossfadein(TRANSITION_DURATION)

    watermark = TextClip(WATERMARK_TEXT, fontsize=30, color='white', bg_color='transparent', method='caption')\
        .set_position(('right', 'bottom')).set_duration(26).set_start(4).set_opacity(0.5)

    closing = TextClip(f"Transform Your Vision\n{AUTHOR_TEXT}\nIG: @craftsandengineering",
                       fontsize=50, color='gold', size=(1280, 720), bg_color='black', method='caption')\
        .set_duration(5).set_start(30).crossfadein(TRANSITION_DURATION)

    video = CompositeVideoClip([
        title,
        clip1.set_start(4),
        clip2.set_start(12),
        clip3.set_start(20),
        watermark,
        closing
    ], size=(1280, 720))

    video.write_videofile(OUTPUT_FILE, fps=24, codec='libx264')
    print(f"Success! Rendering complete: {OUTPUT_FILE}")
    files.download(OUTPUT_FILE)

except Exception as e:
    print(f"An error occurred: {e}")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
imagemagick is already the newest version (8:6.9.11.60+dfsg-1.3ubuntu0.22.04.5).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Moviepy - Building video hough_transform_demo.mp4.
Moviepy - Writing video hough_transform_demo.mp4



Moviepy - Done !
Moviepy - video ready hough_transform_demo.mp4
Success! Rendering complete: hough_transform_demo.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Project README: Hough Transform Video Animation

## Overview
This project generates a parametric video demonstration of the Hough Transform for Line Detection. It takes a high-resolution input image, processes it through edge detection and line extraction algorithms, and compiles the results into a 35-second video with cinematic transitions and professional branding.

## Theoretical Brief: The Hough Transform
The Hough Transform is a feature extraction technique used in image analysis, computer vision, and digital image processing. The purpose of the technique is to find imperfect instances of objects within a certain class of shapes by a voting procedure.

### How it Works
1. **Edge Detection**: First, an edge detector (like Canny) is applied to find the boundaries of objects in the image.
2. **Parameter Space**: In the Cartesian space (x, y), a line can be represented as y = mx + c. However, vertical lines result in an infinite slope. To solve this, the Hough Transform uses the Polar coordinate system where a line is represented as:
   ́́́
   r = x * cos(θ) + y * sin(θ)
   ́́́
   where 'r' is the perpendicular distance from the origin to the line, and 'θ' is the angle formed by this perpendicular line with the horizontal axis.
3. **Accumulator Voting**: Every edge point (x, y) is transformed into a sinusoidal curve in the (r, θ) parameter space. Where multiple curves intersect, it indicates a line passes through those corresponding points in the image space. The intersections with the most votes are identified as the strongest lines.

## Implementation Details
- **Canny Edge Detection**: Used to reduce the image to its structural outlines.
- **Probabilistic Hough Transform**: A more efficient version that analyzes a random subset of points, sufficient for line detection while reducing computation time.
- **MoviePy Integration**: Handles the sequence of clips, cross-fades, and text rendering.

## Credits
- **Author**: Mugambi Ndwiga
- **Organization**: craftsandengineering
- **Platform**: Python / Google Colab